# 10 — Building Interactive Data Science Dashboards

> **📓 Notebook · Module 06 · Intermediate**
> *Transform a Jupyter notebook into a production-quality interactive application.*

---

## 🎯 Learning Objectives

By the end of this notebook you will be able to:

1. Design a dashboard architecture that separates data processing from presentation.
2. Build filter panels that update all dashboard components reactively.
3. Create KPI metric rows that communicate insights at a glance.
4. Combine charts, tables, and controls into a cohesive layout.
5. Implement user feedback for loading states, errors, and empty results.
6. Apply progressive disclosure to manage information density.

## 📋 Prerequisites

- Completed [Notebook 09 — File Upload & Processing](09_file_upload_and_processing.ipynb)
- Pandas basics (DataFrame operations, groupby, merge)
- Plotly basics (px.line, px.bar, px.scatter)
- Understanding of Streamlit widgets, layout, and session state

---

## 📚 Concept: From Notebook to Dashboard

A Jupyter notebook is a **linear narrative** for one analyst.
A Streamlit dashboard is an **interactive application** for many users.

```
NOTEBOOK:  Cell 1 → Cell 2 → Cell 3 → Done
DASHBOARD: User clicks filter → All components update → User explores
```

**The key insight:** In a dashboard, every user action triggers a **rerun**. Your code must be idempotent — it should produce the same result regardless of how many times it runs.

## 🧠 Intuition: The Dashboard Is a Conversation

Think of a dashboard as a **conversation** between the user and the data:

1. **User asks a question** — "Show me sales in the West region"
2. **Dashboard answers** — filters update, charts redraw, metrics recalculate
3. **User follows up** — "Now compare to last year"
4. **Dashboard adapts** — new comparison appears

Your job is to make this conversation **natural and fast**.

In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px

st.set_page_config(page_title="Notebook 10 — Interactive Dashboard", layout="wide")

---

## 🔧 Step 1: Generate a Meaningful Dataset

We'll build an **E-Commerce Sales Explorer**. This dataset represents daily sales across products, regions, and customer segments.

In [ ]:
@st.cache_data
def generate_ecommerce_data(n_rows=2000):
    """Generate realistic e-commerce sales data."""
    np.random.seed(42)

    dates = pd.date_range("2024-01-01", periods=365, freq="D")
    products = ["Laptop", "Phone", "Tablet", "Headphones", "Monitor", "Keyboard"]
    regions = ["North America", "Europe", "Asia Pacific", "Latin America"]
    segments = ["Consumer", "Corporate", "Home Office"]

    data = []
    for _ in range(n_rows):
        date = np.random.choice(dates)
        product = np.random.choice(products)
        region = np.random.choice(regions)
        segment = np.random.choice(segments)

        # Base price varies by product
        base_prices = {
            "Laptop": 1200, "Phone": 800, "Tablet": 500,
            "Headphones": 150, "Monitor": 400, "Keyboard": 80
        }
        unit_price = base_prices[product] * np.random.uniform(0.8, 1.2)
        quantity = np.random.randint(1, 10)
        revenue = unit_price * quantity
        cost = revenue * np.random.uniform(0.4, 0.7)

        data.append({
            "date": date,
            "product": product,
            "region": region,
            "segment": segment,
            "unit_price": round(unit_price, 2),
            "quantity": quantity,
            "revenue": round(revenue, 2),
            "cost": round(cost, 2),
            "profit": round(revenue - cost, 2),
        })

    return pd.DataFrame(data)

df = generate_ecommerce_data()

st.write(f"**Dataset:** {len(df):,} rows × {len(df.columns)} columns")
st.dataframe(df.head(10), use_container_width=True)

---

## 🔧 Step 2: Data Functions (No Streamlit Calls!)

These are **pure functions** — they take data in, return data out. No `st.*` calls.

In [ ]:
def filter_data(df, regions, products, segments, date_range):
    """Apply filters to the DataFrame. Pure function — no st.* calls."""
    result = df.copy()

    if regions:
        result = result[result["region"].isin(regions)]
    if products:
        result = result[result["product"].isin(products)]
    if segments:
        result = result[result["segment"].isin(segments)]
    if date_range and len(date_range) == 2:
        start, end = pd.Timestamp(date_range[0]), pd.Timestamp(date_range[1])
        result = result[(result["date"] >= start) & (result["date"] <= end)]

    return result


def compute_kpis(df):
    """Compute KPI values. Returns a dict."""
    total_revenue = df["revenue"].sum()
    total_profit = df["profit"].sum()
    total_orders = len(df)
    avg_order_value = total_revenue / total_orders if total_orders > 0 else 0
    profit_margin = (total_profit / total_revenue * 100) if total_revenue > 0 else 0
    unique_products = df["product"].nunique()

    return {
        "total_revenue": total_revenue,
        "total_profit": total_profit,
        "total_orders": total_orders,
        "avg_order_value": avg_order_value,
        "profit_margin": profit_margin,
        "unique_products": unique_products,
    }


st.success("✅ Data functions defined — no Streamlit calls inside!")
st.write("These functions can be tested independently, reused in notebooks, or called from APIs.")

---

## 🔧 Step 3: Sidebar Filters

Filters **always** go in the sidebar. The main content area is for results.

In [ ]:
with st.sidebar:
    st.header("🔍 Filters")

    # Date range
    date_range = st.date_input(
        "Date range",
        value=(df["date"].min(), df["date"].max()),
        help="Select the time period to analyze",
    )

    # Region filter
    regions = st.multiselect(
        "Region",
        options=df["region"].unique().tolist(),
        default=df["region"].unique().tolist(),
        help="Filter by geographic region",
    )

    # Product filter
    products = st.multiselect(
        "Product",
        options=df["product"].unique().tolist(),
        default=df["product"].unique().tolist(),
        help="Filter by product category",
    )

    # Segment filter
    segments = st.multiselect(
        "Customer Segment",
        options=df["segment"].unique().tolist(),
        default=df["segment"].unique().tolist(),
        help="Filter by customer type",
    )

    st.divider()
    st.header("⚙️ Display Options")
    chart_type = st.radio("Chart type", ["Line", "Bar", "Area"], horizontal=True)
    show_trend = st.checkbox("Show 7-day moving average", value=True)

---

## 🔧 Step 4: Apply Filters & Handle Empty State

Always check if filtered data is empty before rendering.

In [ ]:
# Apply filters
filtered = filter_data(df, regions, products, segments, date_range)

# Handle empty state
if filtered.empty:
    st.warning(
        "🔍 **No data matches your filters.**\n\n"
        "Try adjusting the date range, regions, or products."
    )
    st.write("**Active filters:**")
    st.write(f"- Regions: {regions}")
    st.write(f"- Products: {products}")
    st.write(f"- Segments: {segments}")
    st.stop()

st.write(f"Showing **{len(filtered):,}** of **{len(df):,}** records")

---

## 🔧 Step 5: KPI Metrics

Lead with insights. Users scan metrics before charts.

In [ ]:
kpis = compute_kpis(filtered)

c1, c2, c3, c4 = st.columns(4)
c1.metric("Total Revenue", f"${kpis['total_revenue']:,.0f}", icon="💰")
c2.metric("Total Profit", f"${kpis['total_profit']:,.0f}", icon="📈")
c3.metric("Total Orders", f"{kpis['total_orders']:,}", icon="📦")
c4.metric("Profit Margin", f"{kpis['profit_margin']:.1f}%", icon="🎯")

st.divider()

---

## 🔧 Step 6: Charts

Use tabs for different views. This prevents visual overload.

In [ ]:
tab_trend, tab_breakdown, tab_segment = st.tabs(
    ["📈 Revenue Trend", "📊 Product Breakdown", "🏢 By Segment"]
)

with tab_trend:
    col_chart, col_summary = st.columns([2, 1])

    with col_chart:
        # Aggregate by date
        daily = filtered.groupby("date").agg({
            "revenue": "sum",
            "profit": "sum",
        }).reset_index()

        if show_trend:
            daily["revenue_ma"] = daily["revenue"].rolling(7, min_periods=1).mean()

        if chart_type == "Line":
            fig = px.line(daily, x="date", y="revenue", title="Daily Revenue")
            if show_trend:
                fig.add_scatter(x=daily["date"], y=daily["revenue_ma"],
                               name="7-day MA", line=dict(dash="dash", color="red"))
        elif chart_type == "Bar":
            fig = px.bar(daily, x="date", y="revenue", title="Daily Revenue")
        else:
            fig = px.area(daily, x="date", y="revenue", title="Daily Revenue")

        fig.update_layout(height=400)
        st.plotly_chart(fig, use_container_width=True)

    with col_summary:
        st.subheader("Summary")
        st.metric("Avg Daily Revenue", f"${daily['revenue'].mean():,.0f}")
        st.metric("Best Day", daily.loc[daily["revenue"].idxmax(), "date"].strftime("%b %d"))
        st.metric("Total Days", len(daily))

with tab_breakdown:
    col_bar, col_pie = st.columns(2)

    with col_bar:
        product_revenue = filtered.groupby("product")["revenue"].sum().sort_values(ascending=True)
        fig_bar = px.bar(x=product_revenue.values, y=product_revenue.index,
                        orientation="h", title="Revenue by Product")
        fig_bar.update_layout(height=400)
        st.plotly_chart(fig_bar, use_container_width=True)

    with col_pie:
        product_profit = filtered.groupby("product")["profit"].sum().reset_index()
        fig_pie = px.pie(product_profit, values="profit", names="product",
                        title="Profit Distribution")
        fig_pie.update_layout(height=400)
        st.plotly_chart(fig_pie, use_container_width=True)

with tab_segment:
    segment_data = filtered.groupby("segment").agg({
        "revenue": "sum",
        "profit": "sum",
        "quantity": "sum",
    }).reset_index()

    fig_segment = px.bar(segment_data, x="segment", y=["revenue", "profit"],
                        barmode="group", title="Revenue & Profit by Segment")
    fig_segment.update_layout(height=400)
    st.plotly_chart(fig_segment, use_container_width=True)

---

## 🔧 Step 7: Data Table with Progressive Disclosure

Use expanders to hide raw data — show it only to users who want it.

In [ ]:
st.divider()

with st.expander("📋 View Filtered Data", expanded=False):
    st.dataframe(
        filtered.sort_values("revenue", ascending=False),
        use_container_width=True,
        hide_index=True,
        column_config={
            "date": st.column_config.DateColumn("Date", format="MMM DD, YYYY"),
            "revenue": st.column_config.NumberColumn("Revenue", format="$%,.2f"),
            "profit": st.column_config.NumberColumn("Profit", format="$%,.2f"),
            "unit_price": st.column_config.NumberColumn("Unit Price", format="$%,.2f"),
        },
        height=400,
    )

with st.expander("📈 Summary Statistics", expanded=False):
    st.dataframe(
        filtered[["revenue", "profit", "quantity", "unit_price"]].describe(),
        use_container_width=True,
    )

---

## 🔧 Step 8: Export

Let users download the filtered data.

In [ ]:
st.divider()

col_dl1, col_dl2, col_spacer = st.columns([1, 1, 2])

with col_dl1:
    st.download_button(
        "📥 Download CSV",
        data=filtered.to_csv(index=False),
        file_name="filtered_sales.csv",
        mime="text/csv",
        use_container_width=True,
    )

with col_dl2:
    import io
    buffer = io.BytesIO()
    with pd.ExcelWriter(buffer, engine="openpyxl") as writer:
        filtered.to_excel(writer, index=False, sheet_name="Sales")
    st.download_button(
        "📥 Download Excel",
        data=buffer.getvalue(),
        file_name="filtered_sales.xlsx",
        use_container_width=True,
    )

---

## ⚠️ Common Mistakes

### Mistake 1: Filters in Main Area

```python
# ❌ Filters clutter the main content
st.selectbox("Region", options)  # In main area

# ✅ Filters belong in the sidebar
with st.sidebar:
    st.selectbox("Region", options)
```

### Mistake 2: No Empty State Handling

```python
# ❌ Crashes or shows confusing output when filtered data is empty
st.dataframe(filtered)  # Shows empty table

# ✅ Always handle empty results
if filtered.empty:
    st.warning("No data matches your filters.")
    st.stop()
```

### Mistake 3: Not Caching Data Loading

```python
# ❌ Reloads on every interaction
def load_data():
    return pd.read_csv("large_file.csv")

# ✅ Cache the result
@st.cache_data
def load_data():
    return pd.read_csv("large_file.csv")
```

---

## 🔍 Debugging Tips

| Symptom | Likely Cause | Fix |
|---|---|---|
| Dashboard shows blank chart | Empty filtered data | Add `filtered.empty` check |
| Sidebar filters don't update chart | Missing `st.rerun()` or state issue | Ensure filters are applied before chart |
| Slow performance | No caching on data loading | Add `@st.cache_data` |
| Chart too narrow | Missing `use_container_width` | Add `use_container_width=True` |
| Numbers not formatted | Raw values displayed | Use f-strings: `f"${val:,.0f}"` |
| Widget values reset on rerun | Wrong widget key usage | Use `key` parameter consistently |

---

## ✏️ Exercises

### Exercise 1: Add a Year-over-Year Comparison
Add a checkbox that, when enabled, compares the current filtered data to a "previous year" dataset. Show the comparison in a side-by-side chart.

### Exercise 2: Add a Region Map
Create a Plotly choropleth or scatter map showing revenue by region. Use `px.scatter_geo` or `px.choropleth`.

### Exercise 3: Add User Feedback
Add `st.spinner()` around the data loading, `st.toast()` when filters are applied, and `st.balloons()` when the user exports data.

## 🚀 Challenge Problem

Build a **Complete Sales Explorer Dashboard** that:
1. Loads the e-commerce dataset with `@st.cache_data`
2. Has sidebar filters for date, region, product, and segment
3. Shows 4 KPI metrics at the top
4. Has 3 tabs: Trend, Breakdown, and Comparison
5. Includes a data table in an expander
6. Provides CSV and Excel download buttons
7. Handles empty states gracefully
8. Uses Plotly for all charts with `use_container_width=True`

Separate data functions from presentation code.

---

## 📌 Key Takeaways

1. **Separate data processing from presentation** — pure functions for data, Streamlit calls only in the UI layer.
2. **Filters go in the sidebar** — main content is for results.
3. **Lead with KPIs** — users scan metrics before charts.
4. **Handle empty states** — always check `filtered.empty` before rendering.
5. **Cache expensive operations** — `@st.cache_data` prevents redundant computation.
6. **Progressive disclosure** — KPIs first, charts second, raw data in expanders.
7. **Consistent formatting** — format numbers, use the same color palette, align metrics.

---

## 📚 Further Reading

- [Streamlit Layouts API](https://docs.streamlit.io/develop/api-reference/layout)
- [Streamlit Metrics API](https://docs.streamlit.io/develop/api-reference/data/st.metric)
- [Streamlit Caching](https://docs.streamlit.io/develop/concepts/architecture/caching)
- [Plotly Express](https://plotly.com/python/plotly-express/)

---

## 🔗 Related Materials

- 📖 Reading: [10 — Building Interactive Data Science Dashboards](../readings/10_interactive_dashboard.md)
- ✏️ Exercise: [10 — Dashboard Workshop](../exercises/10_dashboard_workshop.py)
- 🖥️ Demo App: [Interactive Data Explorer](../apps/10_interactive_data_explorer.py)
- 🚀 Project: [P02 — Data Explorer](../projects/P02_data_explorer.md)